In [1]:
import pandas as pd

df = pd.read_csv("../data/city_day.csv")

print("Shape:", df.shape)
df.head()

Shape: (29531, 16)


,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Ahmedabad,2015-01-01,NaN,NaN,0.92,18.22,17.15,NaN,0.92,27.64,133.36,0.00,0.02,0.00,NaN,NaN
1,Ahmedabad,2015-01-02,NaN,NaN,0.97,15.69,16.46,NaN,0.97,24.55,34.06,3.68,5.50,3.77,NaN,NaN
2,Ahmedabad,2015-01-03,NaN,NaN,17.40,19.30,29.70,NaN,17.40,29.07,30.70,6.80,16.40,2.25,NaN,NaN
3,Ahmedabad,2015-01-04,NaN,NaN,1.70,18.48,17.97,NaN,1.70,18.59,36.08,4.43,10.14,1.00,NaN,NaN
4,Ahmedabad,2015-01-05,NaN,NaN,22.10,21.42,37.76,NaN,22.10,39.33,39.31,7.01,18.89,2.78,NaN,NaN


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 29531 entries, 0 to 29530
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   City        29531 non-null  str    
 1   Date        29531 non-null  str    
 2   PM2.5       24933 non-null  float64
 3   PM10        18391 non-null  float64
 4   NO          25949 non-null  float64
 5   NO2         25946 non-null  float64
 6   NOx         25346 non-null  float64
 7   NH3         19203 non-null  float64
 8   CO          27472 non-null  float64
 9   SO2         25677 non-null  float64
 10  O3          25509 non-null  float64
 11  Benzene     23908 non-null  float64
 12  Toluene     21490 non-null  float64
 13  Xylene      11422 non-null  float64
 14  AQI         24850 non-null  float64
 15  AQI_Bucket  24850 non-null  str    
dtypes: float64(13), str(3)
memory usage: 3.6 MB


In [3]:
df.isnull().sum().sort_values(ascending=False)

Xylene        18109
PM10          11140
NH3           10328
Toluene        8041
Benzene        5623
AQI            4681
AQI_Bucket     4681
PM2.5          4598
NOx            4185
O3             4022
SO2            3854
NO2            3585
NO             3582
CO             2059
Date              0
City              0
dtype: int64

In [4]:
df["AQI_Bucket"].value_counts(dropna=False)

AQI_Bucket
Moderate        8829
Satisfactory    8224
NaN             4681
Poor            2781
Very Poor       2337
Good            1341
Severe          1338
Name: count, dtype: int64

In [5]:
df = df.dropna(subset=["AQI_Bucket"]).copy()

print("Rows after removing missing labels:", len(df))
print(df["AQI_Bucket"].value_counts())

Rows after removing missing labels: 24850
AQI_Bucket
Moderate        8829
Satisfactory    8224
Poor            2781
Very Poor       2337
Good            1341
Severe          1338
Name: count, dtype: int64


In [6]:
# Target: what the model should predict
y = df["AQI_Bucket"]

# Inputs: information the model can use
X = df.drop(columns=["AQI_Bucket", "AQI", "Date"])

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nInput columns:")
print(X.columns.tolist())

X shape: (24850, 13)
y shape: (24850,)

Input columns:
['City', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene']


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("\nTraining class proportions:")
print(y_train.value_counts(normalize=True).round(3))
print("\nTest class proportions:")
print(y_test.value_counts(normalize=True).round(3))

Training rows: 19880
Test rows: 4970

Training class proportions:
AQI_Bucket
Moderate        0.355
Satisfactory    0.331
Poor            0.112
Very Poor       0.094
Good            0.054
Severe          0.054
Name: proportion, dtype: float64

Test class proportions:
AQI_Bucket
Moderate        0.355
Satisfactory    0.331
Poor            0.112
Very Poor       0.094
Severe          0.054
Good            0.054
Name: proportion, dtype: float64


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

numeric_features = X_train.select_dtypes(include="number").columns
categorical_features = ["City"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed training shape: (19880, 38)
Processed test shape: (4970, 38)


In [9]:
import numpy as np

classes = np.sort(y_train.unique())
class_to_id = {label: i for i, label in enumerate(classes)}

y_train_encoded = y_train.map(class_to_id).to_numpy()
y_test_encoded = y_test.map(class_to_id).to_numpy()

print("Class mapping:", class_to_id)
print("Encoded training shape:", y_train_encoded.shape)

Class mapping: {'Good': 0, 'Moderate': 1, 'Poor': 2, 'Satisfactory': 3, 'Severe': 4, 'Very Poor': 5}
Encoded training shape: (19880,)


In [14]:
from AQC_model import SoftmaxRegression

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

ModuleNotFoundError: No module named 'AQC_model'